# Prices: S&P 500 features, labels, and Fed rate

Downloads daily S&P 500 data from Yahoo Finance and the Fed's target rate from FRED, and combines them into one table with one row per trading day from Jan 3, 2022 to Sep 30, 2026.

- **log_return**: the log return of the S&P 500
- **hl_range**: how much the index moves within the day, as a fraction of the price
- **log_volume_change**: how much the volume changes from the previous day
- **label**: 1 if tomorrow's close is higher than today's, else 0
- **rate_upper**: upper bound of the Fed's target rate (FRED series DFEDTARU) in effect after any decision that day. Used to pick which Kalshi markets to read.

Results are saved to `data/prices.csv`.

From Yahoo Finance (yfinance, ticker ^GSPC) and FRED

In [2]:
# import dependencies
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

## Download S&P 500
Starts one day early so the first day has a previous close and one day later so last day has a label. 

In [3]:
# download S&P 500 data from Yahoo Finance
raw = yf.download("^GSPC", start="2021-12-31", end="2026-10-02", progress=False, auto_adjust=False)
raw.columns = raw.columns.get_level_values(0) # flatten yfinance's two-level column names

## Features and label
Add log_return, high-low range, log_volume chnage, and label to data

In [4]:
df = pd.DataFrame(index=raw.index)

# get log returns
df["log_return"] = np.log(raw["Close"] / raw["Close"].shift(1))
#hl_range: how much the index moves within the day, as a fraction of the price. calm = small and volatile = big range.
df["hl_range"] = (raw["High"] - raw["Low"]) / raw["Close"] # come from same day, so no need to shift    

#log_volume_change: how much the volume changes from the previous day
with np.errstate(divide="ignore"):
    df["log_volume_change"] = np.log(raw["Volume"] / raw["Volume"].shift(1))
df["log_volume_change"] = df["log_volume_change"].replace([np.inf, -np.inf], 0.0)
df["label"] = (raw["Close"].shift(-1) > raw["Close"]).astype(int) # label: 1 if tomorrow's close is higher than today's, else 0

df = df.loc["2022-01-03":"2026-09-30"] # trim to the first and last trading days of 2022-2026

## Fed target rate
Add fed rate data
FRED records a rate change from the day after the decision, so each trading day takes the next calendar day's value.

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFEDTARU"
fed_rate = pd.read_csv(url, index_col=[0], parse_dates=[0])
fed_rate = fed_rate.iloc[:, 0]
df["rate_upper"] = fed_rate.reindex(df.index + pd.Timedelta(days=1), method="ffill").values # every trading day shifted forward by 1 day

## Checks

In [7]:
print("Trading days:", len(df))     # 1190
print("Missing values:", int(df.isna().sum().sum()))  # 0
print("Up-day share:", round(df["label"].mean(), 3)) # 52.8%
print(df["rate_upper"][df["rate_upper"].diff() != 0]) # show the days when the Fed rate changed

Trading days: 1190
Missing values: 0
Up-day share: 0.528
Date
2022-01-03    0.25
2022-03-16    0.50
2022-05-04    1.00
2022-06-15    1.75
2022-07-27    2.50
2022-09-21    3.25
2022-11-02    4.00
2022-12-14    4.50
2023-02-01    4.75
2023-03-22    5.00
2023-05-03    5.25
2023-07-26    5.50
2024-09-18    5.00
2024-11-07    4.75
2024-12-18    4.50
2025-09-17    4.25
2025-10-29    4.00
2025-12-10    3.75
2026-09-16    4.00
Name: rate_upper, dtype: float64


## Save data
Creates a new data folder if one doesn't exist yet and results are saved to `prices.csv` in data folder.

In [9]:
Path("data").mkdir(exist_ok=True) # check if the "data" directory exists, and create it if it doesn't
df.to_csv("data/prices.csv") # write the dataframe to a CSV file
print("Saved", df.shape, "prices to data/prices.csv")

Saved (1190, 5) prices to data/prices.csv
